# 04_ScanVisitSummaries: image-quality indicators of all visits of a stage-1 run, by observation type and reason

No star matching here: only the exposure records and `preliminary_visit_summary` (one small table per visit).

Questions:
1. Which `observation_type` / `observation_reason` do the visits of the run carry? (the new BPS query keeps `exposure.observation_type='science'`; do `infocus` or `acq` exposures survive that filter?)
2. Do the PSF-model failures seen in notebook 03 (`shapeletsIqScore` of 0.1-0.2 against < 0.007 for clean visits, large `psfStarDeltaE1/E2Median`) concentrate in some types or reasons?
3. What fraction of the visits of each band is affected?

Run with the `lsst` kernel at CC-IN2P3. Start with `STRIDE = 10`, then set `STRIDE = 1` for the full scan.

- author : Sylvie Dagoret-Campagne
- creation date : 2026-11-04
- last update : 2026-11-04 : use input collection : `u/dagoret/LSSTCam/runcosmos01_20261026_tract9813patch50_RunCosmosDemo/stage1`
- last update : 2026-11-05 : use input collection : `u/dagoret/LSSTCam/runcosmos01_20261103_tract9813patchsel_RunCosmos9patches/stage1`
- `DP2 tutorial` : https://dp2.lsst.io/tutorials/notebook/204/notebook-204-2.html
- `quantum graph for step1a` : https://tigress-web.princeton.edu/~lkelvin/pipelines/2026/w_2026_43/drp_pipe/LSSTCam/DRP/pipeline_drp_pipe_LSSTCam_DRP_step1a-single-visit-detectors.pdf

In [ ]:
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SRC = Path('../../src').resolve()
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from rubincc import monster as mon

NB_TAG = '04_ScanVisitSummaries'
DATA_DIR = Path(f'data_{NB_TAG}')
FIG_DIR = Path(f'figs_{NB_TAG}')
DATA_DIR.mkdir(exist_ok=True)
FIG_DIR.mkdir(exist_ok=True)

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['axes.labelsize'] = 'x-large'
plt.rcParams['axes.titlesize'] = 'x-large'
plt.rcParams['xtick.labelsize'] = 'x-large'
plt.rcParams['ytick.labelsize'] = 'x-large'


def savefig(fig, name):
    # every figure is saved as PDF and PNG
    for ext in ('pdf', 'png'):
        fig.savefig(FIG_DIR / f'{NB_TAG}_{name}.{ext}', bbox_inches='tight', dpi=150)

In [ ]:
REPO = 'main'
USER = os.getenv('USER')

#RUN = f'u/{USER}/LSSTCam/runcosmos01_20261026_tract9813patch50_RunCosmosDemo/stage1'
RUN = f'u/{USER}/LSSTCam/runcosmos01_20261103_tract9813patchsel_RunCosmos9patches/stage1'
RUN_COLLECTION = RUN   # collection to scan (switch to NEW_RUN once the new run has produced outputs)

STRIDE = 10       # 1 = all visits; 10 = one visit out of ten (quick first look)
IQ_CUT = 0.05     # shapeletsIqScore above this value = PSF model failure (clean visits of notebook 03: < 0.007)
E_CUT = 0.02      # |psfStarDeltaE1Median| or |psfStarDeltaE2Median| above this value = failure

## 1. Visits and exposure records

`observation_type` and `observation_reason` are fields of the exposure records (the visit records of notebook 02 returned `None` for `observation_type`). For LSSTCam the visit id equals the exposure id.

In [ ]:
butler = mon.get_butler(REPO, collections=RUN_COLLECTION)
refs = butler.query_datasets('preliminary_visit_summary', limit=20000)
all_visits = sorted({ref.dataId['visit'] for ref in refs})
print(len(all_visits), 'visits with preliminary_visit_summary in', RUN_COLLECTION)
visits = all_visits[::STRIDE]
print('visits scanned:', len(visits))

records = butler.registry.queryDimensionRecords(
    'exposure', where='instrument = inst AND exposure IN (ids)', bind={'inst': 'LSSTCam', 'ids': visits})
exp_df = pd.DataFrame([{
    'visit': rec.id,
    'observation_type': getattr(rec, 'observation_type', None),
    'observation_reason': getattr(rec, 'observation_reason', None),
    'science_program': getattr(rec, 'science_program', None),
    'target_name': getattr(rec, 'target_name', None),
    'day_obs': getattr(rec, 'day_obs', None),
} for rec in records])
print(exp_df.shape)
print(exp_df.groupby(['observation_type', 'observation_reason']).size().rename('n_visits').to_string())

## 2. Scan of `preliminary_visit_summary`

In [ ]:
KEY = ['zeroPoint', 'expTime', 'psfSigma', 'skyBg', 'nPsfStar', 'astromOffsetMean', 'zenithDistance', 'effTime',
       'magLim', 'shapeletsIqScore', 'psfStarDeltaE1Median', 'psfStarDeltaE2Median', 'psfStarDeltaSizeMedian',
       'psfApFluxDelta', 'psfApCorrSigmaScaledDelta', 'psfTraceRadiusDelta']

rows = []
failures = []
for n, visit in enumerate(visits):
    try:
        vs = mon.as_dataframe(butler.get('preliminary_visit_summary', visit=int(visit), instrument='LSSTCam'))
        row = {'visit': int(visit), 'n_det': len(vs), 'band': str(vs['band'].iloc[0]) if 'band' in vs.columns else None}
        for col in KEY:
            if col in vs.columns:
                row[col] = float(np.nanmedian(vs[col]))  # median over the detectors of the visit
        rows.append(row)
    except Exception as exc:
        failures.append((visit, repr(exc)))
    if (n + 1) % 100 == 0 or n + 1 == len(visits):
        print(f'{n + 1}/{len(visits)} visits read, {len(failures)} failures')

print('first failures:', failures[:3])
scan = pd.DataFrame(rows).merge(exp_df, on='visit', how='left')
scan['zeroPoint_30s'] = scan['zeroPoint'] - 2.5 * np.log10(scan['expTime'] / 30.0)
scan['iq_bad'] = scan['shapeletsIqScore'] > IQ_CUT
scan['e_bad'] = (scan['psfStarDeltaE1Median'].abs() > E_CUT) | (scan['psfStarDeltaE2Median'].abs() > E_CUT)
scan.to_csv(DATA_DIR / f'{NB_TAG}_scan.csv', index=False)
print(scan.shape)

## 3. Failure fractions

In [ ]:
# by observation type and reason: fraction of visits with a PSF-model failure
by_reason = (scan.groupby(['observation_type', 'observation_reason'])
             .agg(n=('visit', 'size'), frac_iq_bad=('iq_bad', 'mean'), frac_e_bad=('e_bad', 'mean'),
                  median_nPsfStar=('nPsfStar', 'median'))
             .round(3))
by_reason.to_csv(DATA_DIR / f'{NB_TAG}_by_reason.csv')
print(by_reason.to_string())

print()
# by band, for the visits that the BPS filter observation_type='science' would keep
sci = scan[scan['observation_type'] == 'science']
by_band = sci.groupby('band').agg(n=('visit', 'size'), frac_iq_bad=('iq_bad', 'mean'), frac_e_bad=('e_bad', 'mean')).round(3)
print(by_band.to_string())

In [ ]:
# which observation reasons survive the filter observation_type='science', and how many look bad?
reasons_sci = (sci.groupby('observation_reason')
               .agg(n=('visit', 'size'), frac_iq_bad=('iq_bad', 'mean'))
               .sort_values('n', ascending=False))
print(reasons_sci.round(3).to_string())

clean_reasons = reasons_sci[reasons_sci['frac_iq_bad'] < 0.02].index.tolist()
quoted = ', '.join("'" + str(r) + "'" for r in clean_reasons)
print()
print('candidate extra clause for the BPS dataQuery (to be checked before use):')
print(f"AND exposure.observation_reason IN ({quoted})")

## 4. Figures

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
bins = np.logspace(-5, 0, 40)
for otype, s in scan.groupby('observation_type'):
    axes[0].hist(np.clip(s['shapeletsIqScore'].dropna(), 1e-5, None), bins=bins, histtype='step', lw=2,
                 label=f'{otype} (N={len(s)})')
axes[0].axvline(IQ_CUT, color='k', ls='--')
axes[0].set_xscale('log')
axes[0].set_xlabel('shapeletsIqScore (median over detectors)')
axes[0].set_ylabel('N visits')
axes[0].legend()

for otype, s in scan.groupby('observation_type'):
    axes[1].plot(s['psfStarDeltaE1Median'], s['psfStarDeltaE2Median'], '.', ms=5, alpha=0.6, label=otype)
axes[1].set_xlim(-0.01, 0.01)
axes[1].set_ylim(-0.01, 0.01)
axes[1].set_xlabel('psfStarDeltaE1Median')
axes[1].set_ylabel('psfStarDeltaE2Median')
axes[1].legend()
plt.tight_layout()
savefig(fig, 'psf_quality_by_observation_type')

In [ ]:
# normalised zero point per band: clean visits against PSF-failure visits
fig, ax = plt.subplots(figsize=(12, 6))
bands = sorted(scan['band'].dropna().unique())
for k, band in enumerate(bands):
    s = scan[scan['band'] == band]
    ok = s[~s['iq_bad']]
    bad = s[s['iq_bad']]
    ax.plot(np.full(len(ok), k) + np.random.uniform(-0.2, 0.2, len(ok)), ok['zeroPoint_30s'], '.', alpha=0.5, color='C0',label="ok")
    ax.plot(np.full(len(bad), k) + np.random.uniform(-0.2, 0.2, len(bad)), bad['zeroPoint_30s'], 'rx',color="r",label="bad")
ax.set_xticks(range(len(bands)))
ax.set_xticklabels(bands)
ax.set_ylabel('zeroPoint normalised to 30 s (median over detectors)')
ax.set_title(f'blue: shapeletsIqScore <= {IQ_CUT}; red crosses: above')
#ax.legend()
savefig(fig, 'zeropoint_by_band')